In [35]:
import pandas as pd

In [36]:
data = pd.read_csv('../data/merged_filter_ingestion.csv')
data = data.drop(columns=['Unnamed: 0', 'Segment', 'Region'])

# Convert Date to datetime
data['Date'] = pd.to_datetime(data['Date'])

# Get unique branches
branches = data['Branch'].unique()
print(f"Found {len(branches)} branches: {branches}")

# Get date range
min_date = data['Date'].min()
max_date = data['Date'].max()
print(f"Date range: {min_date} to {max_date}")

Found 5 branches: ['COK' 'MAA' 'SBD' 'BLR' 'SBD1']
Date range: 2019-04-03 00:00:00 to 2025-06-30 00:00:00


In [37]:
data['Tonnage'].value_counts()

Tonnage
1.5    95274
1.0    49565
1.8    25884
2.2    11833
0.8     6779
Name: count, dtype: int64

In [32]:
# Aggregate quantity by Date and Branch
# If there are multiple rows for same date/branch, sum them up
agg_data = data.groupby(['Date', 'Branch'])['Quantity'].sum().reset_index()
agg_data.rename(columns={'Quantity': 'qty'}, inplace=True)

print(f"Aggregated data shape: {agg_data.shape}")
print(f"Unique (Date, Branch) combinations before filling: {len(agg_data)}")


Aggregated data shape: (8519, 3)
Unique (Date, Branch) combinations before filling: 8519


In [33]:
# Create complete date range for each branch
complete_dates = pd.date_range(start=min_date, end=max_date, freq='D')
complete_df = pd.DataFrame({
    'Date': complete_dates
})

# Create cartesian product of all dates and all branches
complete_df = complete_df.assign(key=1).merge(
    pd.DataFrame({'Branch': branches, 'key': 1}), 
    on='key'
).drop('key', axis=1)

print(f"Complete date-branch combinations: {len(complete_df)}")
print(f"Expected: {len(complete_dates)} dates × {len(branches)} branches = {len(complete_dates) * len(branches)}")


Complete date-branch combinations: 11405
Expected: 2281 dates × 5 branches = 11405


In [34]:
# Merge with actual data, filling missing dates with qty = 0
data_filled = complete_df.merge(agg_data, on=['Date', 'Branch'], how='left')
data_filled['qty'] = data_filled['qty'].fillna(0)

print(f"Final data shape: {data_filled.shape}")
print(f"Rows with qty = 0: {(data_filled['qty'] == 0).sum()}")
print(f"Rows with qty > 0: {(data_filled['qty'] > 0).sum()}")

# Verify each branch has all dates
for branch in branches:
    branch_dates = data_filled[data_filled['Branch'] == branch]['Date'].nunique()
    expected_dates = len(complete_dates)
    if branch_dates == expected_dates:
        print(f"✓ {branch}: Has all {branch_dates} dates")
    else:
        print(f"✗ {branch}: Missing dates! Has {branch_dates}, expected {expected_dates}")

data = data_filled
print("\nData updated with all missing dates filled with qty = 0")


Final data shape: (11405, 3)
Rows with qty = 0: 2886
Rows with qty > 0: 8519
✓ COK: Has all 2281 dates
✓ MAA: Has all 2281 dates
✓ SBD: Has all 2281 dates
✓ BLR: Has all 2281 dates
✓ SBD1: Has all 2281 dates

Data updated with all missing dates filled with qty = 0


In [ ]:
# requirements: pandas, numpy, lightgbm, sklearn
import pandas as pd
import numpy as np
from sklearn.model_selection import TimeSeriesSplit
import lightgbm as lgb
from sklearn.metrics import mean_absolute_error

# 1) load
df = pd.read_csv('sales.csv', parse_dates=['Date'])   # your table
df['Month'] = df['Date'].dt.to_period('M').dt.to_timestamp()  # month start
# 2) aggregate monthly per Branch x Tonnage
monthly = (df.groupby(['Branch','Tonnage','Month'])
             .agg(Quantity=('Quantity','sum'),
                  avg_star=('Star-Rating','mean'))
             .reset_index())

# 3) create full grid (fill missing months)
min_date = monthly['Month'].min()
max_date = monthly['Month'].max()
months = pd.date_range(min_date, max_date, freq='MS')
branches = monthly['Branch'].unique()
tonnages = monthly['Tonnage'].unique()

grid = pd.MultiIndex.from_product([branches, tonnages, months], names=['Branch','Tonnage','Month']).to_frame(index=False)
monthly = grid.merge(monthly, on=['Branch','Tonnage','Month'], how='left').sort_values(['Branch','Tonnage','Month'])
monthly['Quantity'] = monthly['Quantity'].fillna(0)  # or np.nan then impute

# 4) feature engineering for each series
def make_features(g):
    g = g.sort_values('Month').copy()
    g['qty_lag_1'] = g['Quantity'].shift(1)
    g['qty_lag_12'] = g['Quantity'].shift(12)
    g['roll_mean_3'] = g['Quantity'].shift(1).rolling(3).mean()
    g['month'] = g['Month'].dt.month
    g['month_sin'] = np.sin(2*np.pi*g['month']/12)
    g['month_cos'] = np.cos(2*np.pi*g['month']/12)
    g['avg_star'] = g['avg_star'].fillna(g['avg_star'].mean())  # if missing
    return g

monthly = monthly.groupby(['Branch','Tonnage']).apply(make_features).reset_index(drop=True)

# 5) drop initial rows with NaNs
monthly = monthly.dropna(subset=['qty_lag_1','qty_lag_12','roll_mean_3'])

# 6) train/val split by time: last 12 months as test
last_date = monthly['Month'].max()
val_start = last_date - pd.DateOffset(months=11)  # validate on last 12 months
train = monthly[monthly['Month'] < val_start]
val = monthly[monthly['Month'] >= val_start]

features = ['qty_lag_1','qty_lag_12','roll_mean_3','month','month_sin','month_cos','avg_star','Tonnage']  # include Branch as categorical if desired

# 7) train LightGBM (example single-step)
dtrain = lgb.Dataset(train[features], label=train['Quantity'], categorical_feature=['Tonnage'])
dval = lgb.Dataset(val[features], label=val['Quantity'], reference=dtrain, categorical_feature=['Tonnage'])

params = {
    'objective':'regression',
    'metric':'mae',
    'learning_rate':0.05,
    'num_leaves':31,
    'verbosity': -1
}

model = lgb.train(params, dtrain, valid_sets=[dtrain,dval], early_stopping_rounds=50, num_boost_round=2000)

# 8) predict and evaluate
pred = model.predict(val[features])
print('MAE:', mean_absolute_error(val['Quantity'], pred))
